# 04 — E4: transferencia FabriD → CISUTAC, sin ajuste

**Pregunta:** un detector entrenado con primeros planos de tela (FabriD), ¿encuentra agujeros y manchas en fotos de prendas usadas (CISUTAC) sin volver a entrenarlo? ¿Cuánto cae respecto de su propio dominio y respecto del detector entrenado en CISUTAC (E1)?

FabriD y CISUTAC no difieren sólo en la tela y la iluminación, sino también en la **escala**: FabriD son primeros planos y CISUTAC, prendas enteras. Por eso, además de la foto completa, se prueban *tiles* de distinto tamaño, que cambian el tamaño con el que el defecto llega al modelo.

| ID | Qué mide | Datos de evaluación | Inferencia |
|---|---|---|---|
| **E4-F** | Antes del cambio de dominio | FabriD valid (test al final), hole y stain por separado | Foto del export (≤ 640 px) entera |
| **E4-C** | Después, con la foto completa | CISUTAC Holes y Spots, valid (test al final) | Lado mayor 640, como el A0 del E1 |
| **E4-T** | Después, con tiles | Ídem | Tiles de 640 o 320 px de la original, que entran al modelo a 640 (los de 320 se amplían ×2) |

La sección 9 es un **anexo exploratorio** (*pendiente de aprobación del grupo*): la misma evaluación con FabricSpotDefect como fuente, sólo para Spots.

**Reglas** (`reports/protocolo_experimental.md`, secciones 2, 5 y 7):

- **Modelo:** YOLO11n preentrenado en COCO y ajustado en FabriD con las clases hole y stain, el split agrupado `reports/fabrid/fab4_split_agrupado.csv` y sin los 19 casos ambiguos (`scripts/prepare_fabrid_yolo.py`). Valid de FabriD elige la época. Si el grupo decide reusar el modelo del E3, alcanza con poner su ruta en `PESOS_FABRID`.
- **Sin ajuste:** los pesos no se entrenan con CISUTAC. En Holes sólo cuentan las predicciones de la clase hole y en Spots, las de stain.
- **Valid de CISUTAC** elige el umbral de confianza y el tamaño de tile, como en el E1 (*pendiente de aprobación del grupo*). **El test se usa una sola vez, al final.**
- **Mismas métricas que el E1**, con `src/detection_eval.py` en coordenadas de la original: mAP@0.5, mAP@0.5:0.95, AP_small, precisión/recall/F1 y recall por tamaño.

> **Cómo correrlo.** Necesita CISUTAC y FabriD v3 en `data/` (ver `docs/DATASETS.md`) y `pip install ultralytics==8.4.209 sahi==0.14.0 pycocotools==2.0.11 pandas`. Primero conviene correrlo con `MODO_PRUEBA = True` (1 época, pocas imágenes). Las corridas quedan en `results/transferencia/` y se reutilizan al volver a ejecutar (salvo `REENTRENAR = True`).

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import cisutac_baseline as cb
import transferencia as tr

cb.RESULTS = ROOT / "results" / "transferencia"   # las corridas del E4 quedan aparte de las del bloque A
pd.set_option("display.max_columns", 30)
print("Datos:", cb.DATA)
print("Resultados:", cb.RESULTS)

## 0. Parámetros

In [ ]:
MODO_PRUEBA = True       # True: 1 época y pocas imágenes, sólo para verificar el pipeline
REENTRENAR = False       # True: ignora corridas guardadas y vuelve a entrenar/evaluar
EVALUAR_TEST = False     # True sólo al final, una única vez, con la configuración ya elegida en valid
ANEXO_FSD = False        # True: anexo exploratorio FabricSpotDefect → Spots (sección 9, pendiente de aprobación)

# Modelo de FabriD (misma receta que el A0 del E1, así la diferencia con el E1 son los datos de entrenamiento)
PESOS_FABRID = None                   # ruta a un best.pt ya entrenado en FabriD (p. ej. el del E3); None = entrenarlo acá
VARIANTE_FABRID = "hole_stain_cajas"  # carpeta data/derived/fabrid_<variante> de scripts/prepare_fabrid_yolo.py
MODELO = "yolo11n.pt"                 # preentrenado en COCO
SEMILLA = 0
EPOCAS, PACIENCIA, BATCH = 100, 30, 16
CONF_EVAL = 0.001                     # confianza mínima de las predicciones guardadas (bajo, para calcular mAP)

# Inferencia en CISUTAC: lado del tile en la foto original; todos entran al modelo a 640
TILES = [640, 320]
SOLAPAMIENTO = 0.2

if MODO_PRUEBA:
    EPOCAS, PACIENCIA = 1, 1
    FRACCION, LIMITE_EVAL = 0.1, 3
else:
    FRACCION, LIMITE_EVAL = 1.0, None
SUFIJO = "_prueba" if MODO_PRUEBA else ""


def pesos(run_dir):
    return Path(run_dir) / "weights" / "best.pt"


def evaluar_fabrid(clave, weights, split="valid", umbral=None):
    # Antes del cambio de dominio: una clase de FabriD por vez, en px del export.
    gt = tr.load_fabrid_ground_truth(split, clave, VARIANTE_FABRID)
    if LIMITE_EVAL:
        gt = tr.balanced_ground_truth_subset(gt, LIMITE_EVAL)
    return cb.evaluate(f"e4_fabrid_{clave}__{split}{SUFIJO}", "fabrid", weights, split=split, mode="completa",
                       imgsz=640, classes=[tr.class_id(weights, clave)], conf=CONF_EVAL, threshold=umbral,
                       gt=gt, image_dir=tr.fabrid_export_dir(), reevaluate=REENTRENAR,
                       confirm_test=split == "test")


def evaluar_cisutac(dataset, weights, modo, tile=640, split="valid", umbral=None, clave=None, prefijo="e4"):
    # Después: sólo la clase de la fuente que corresponde al dataset (hole o stain de FabriD; spot en el anexo),
    # en px de la original.
    detalle = "completa640" if modo == "completa" else f"tiles{tile}_o{round(100 * SOLAPAMIENTO)}"
    return cb.evaluate(f"{prefijo}_{dataset}__{split}_{detalle}{SUFIJO}", dataset, weights, split=split, mode=modo,
                       imgsz=640, tile=tile, overlap=SOLAPAMIENTO, tile_imgsz=640,
                       classes=[tr.class_id(weights, clave or tr.CLASE_FABRID[dataset])], conf=CONF_EVAL,
                       threshold=umbral, limit=LIMITE_EVAL, reevaluate=REENTRENAR, confirm_test=split == "test")


def evaluar_fsd(weights, split="valid", umbral=None):
    # Anexo: FabricSpotDefect en su propio dominio, en px de la foto tal como está guardada.
    return cb.evaluate(f"e4fsd_fsd_spot__{split}{SUFIJO}", "fsd", weights, split=split, mode="completa", imgsz=640,
                       classes=[tr.class_id(weights, "spot")], conf=CONF_EVAL, threshold=umbral,
                       gt=tr.load_ground_truth(tr.fsd_derived_dir(), split, "spot"),
                       image_dir=tr.fsd_original_dir(), limit=LIMITE_EVAL, reevaluate=REENTRENAR,
                       confirm_test=split == "test")


def mejor_en_valid(evals):
    # Mismo criterio que el E1: mayor mAP@0.5 en valid y, si empatan, la opción más barata.
    return max(evals, key=lambda m: (m["mAP@0.5"], -m["segundos_por_imagen"]))

In [ ]:
import importlib.metadata as md
for paquete in ("ultralytics", "sahi", "pycocotools", "torch", "pandas"):
    try:
        print(f"{paquete:12s} {md.version(paquete)}")
    except md.PackageNotFoundError:
        print(f"{paquete:12s} FALTA -> pip install {paquete}")
print("Dispositivo:", cb.pick_device())

## 1. Escala: con qué tamaño llega cada defecto al modelo

Lado equivalente mediano (√(ancho·alto), en px) con el que el defecto entra a la red. FabriD entra entero, porque el export ya tiene como máximo 640 px; en CISUTAC depende de cómo se infiere. Sale de las tablas del EDA (`reports/fabrid/fab3_tamanos_clase.csv` y `reports/cisutac/instancias_original.csv`), sin ningún modelo.

In [ ]:
tr.scale_table(TILES)

- **Con la foto completa, los defectos llegan unas 8 veces (agujeros) y 20 veces (manchas) más chicos** que los que el modelo vio en FabriD. Es el mismo problema del A0 del E1, sumado al cambio de dominio.
- **Con tiles de 640**, los agujeros quedan a una escala parecida a la de FabriD (25 frente a 32 px); las manchas siguen unas 3 veces más chicas.
- **Con tiles de 320** ampliados a 640, las manchas se acercan (36 frente a 60 px) y los agujeros pasan a ser algo más grandes que en FabriD (49 frente a 32 px), con unas 4 veces más tiles por foto.

Por eso se espera que la elección del tile no sea la misma para Holes y para Spots; se decide con valid.

## 2. Datos de FabriD

`scripts/prepare_fabrid_yolo.py --clases hole stain --formato cajas` arma enlaces al export (no copia imágenes), labels con una caja por defecto, `data.yaml` y cajas COCO por clase para evaluar. Entran las normales concordantes, como fondo, y las imágenes cuyas instancias son todas hole o stain. Se excluyen los 19 ambiguos, las imágenes que sólo tienen otras clases y las mixtas (*pendiente de aprobación del grupo*: incluirlas con `--mixtas` dejaría como fondo defectos reales de otra clase). Al dejar afuera las marcas de aguja también desaparece el atajo de las imágenes de 512×512.

In [ ]:
datos_fabrid = tr.fabrid_derived_dir(VARIANTE_FABRID)
if not (datos_fabrid / "data.yaml").exists():
    if not (tr.fabrid_export_dir() / "data.yaml").exists():
        raise FileNotFoundError(f"Falta FabriD v3 en {tr.fabrid_export_dir()} (ver docs/DATASETS.md)")
    subprocess.run([sys.executable, str(ROOT / "scripts" / "prepare_fabrid_yolo.py"),
                    "--clases", "hole", "stain", "--formato", "cajas"], check=True)

resumen = json.loads((datos_fabrid / "resumen.json").read_text())
print("Excluidas:", resumen["excluidas"])
pd.DataFrame(resumen["splits"]).T

## 3. Modelo entrenado en FabriD

YOLO11n de COCO con la receta del A0 (aumentos por defecto de Ultralytics, `imgsz` 640): así la diferencia con el E1 son los datos de entrenamiento. Valid de FabriD elige la mejor época; CISUTAC no interviene.

In [ ]:
if PESOS_FABRID is None:
    run_fabrid = cb.train(f"e4_fabrid_hole_stain_yolo11n_640{SUFIJO}", datos_fabrid / "data.yaml", imgsz=640,
                          epochs=EPOCAS, patience=PACIENCIA, batch=BATCH, seed=SEMILLA, model=MODELO,
                          retrain=REENTRENAR, fraction=FRACCION)
    PESOS = pesos(run_fabrid)
    cb.plot_curves({"FabriD hole + stain": run_fabrid})
else:
    PESOS = Path(PESOS_FABRID)
print("Pesos:", PESOS)

## 4. Antes del cambio de dominio: FabriD valid

Cada clase se evalúa por separado sobre todas las imágenes de valid de la variante: una predicción de hole sobre una imagen con una mancha cuenta como falso positivo. Las cajas están en px del export, así que **AP_small y el recall por tamaño no son comparables con los de CISUTAC**: cambian las fotos y la escala.

In [ ]:
evals_fabrid = [evaluar_fabrid(clave, PESOS) for clave in ("hole", "stain")]
display(cb.metrics_table(evals_fabrid))
display(cb.size_table(evals_fabrid))

## 5. Después del cambio de dominio: CISUTAC valid

Mismo modelo, sin ajuste, con la foto completa y con cada tamaño de tile. Los tiles se fusionan con la misma NMS que el E1 (SAHI, IoU 0,5), y la clase se filtra antes de fusionar.

In [ ]:
evals_cisutac = {dataset: [evaluar_cisutac(dataset, PESOS, "completa")]
                          + [evaluar_cisutac(dataset, PESOS, "tiles", tile=tile) for tile in TILES]
                 for dataset in ("holes", "spots")}
for dataset, evals in evals_cisutac.items():
    print(dataset.upper())
    display(cb.metrics_table(evals))
    display(cb.size_table(evals))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8), gridspec_kw={"width_ratios": [2, 1]})
for row, (dataset, evals) in zip(axes, evals_cisutac.items()):
    cb.plot_recall_by_size(evals, ax=row[0])
    row[0].set_title(f"CISUTAC {dataset}: recall por lado equivalente (px de la original)")
    cb.plot_cost_vs_quality(evals, metric="mAP@0.5", ax=row[1])
plt.tight_layout()

## 6. Ejemplos

Recortes alrededor de defectos reales de valid, con las predicciones de la mejor configuración de cada dataset sobre su umbral. Sirven para distinguir si el modelo falla por escala (no ve el defecto) o por dominio (confunde partes de la prenda con defectos).

In [ ]:
for dataset, evals in evals_cisutac.items():
    cb.plot_examples(dataset, mejor_en_valid(evals)["corrida"], n=4)

## 7. Selección en valid y evaluación única en test

**Criterio fijado antes de mirar test:** en cada dataset de CISUTAC se elige el tamaño de tile con mayor mAP@0.5 en valid (si empatan, el más barato) y se evalúa en test junto con la foto completa, cada uno con su umbral de valid. En FabriD, cada clase se evalúa en test con el umbral de su valid. Es el mismo criterio que el E1.

In [ ]:
elegidas = {dataset: mejor_en_valid([m for m in evals if m["modo"] == "tiles"])
            for dataset, evals in evals_cisutac.items()}
for dataset, m in elegidas.items():
    print(f"{dataset}: tile elegido {m['tile']} (mAP@0.5 en valid {m['mAP@0.5']:.3f})")

if EVALUAR_TEST:
    test_fabrid = [evaluar_fabrid(clave, PESOS, split="test", umbral=m["umbral_confianza"])
                   for clave, m in zip(("hole", "stain"), evals_fabrid)]
    test_cisutac = {dataset: [evaluar_cisutac(dataset, PESOS, m["modo"], tile=m["tile"], split="test",
                                              umbral=m["umbral_confianza"])
                              for m in (evals_cisutac[dataset][0], elegidas[dataset])]
                    for dataset in evals_cisutac}
    display(cb.metrics_table(test_fabrid))
    for dataset, evals in test_cisutac.items():
        print(dataset.upper())
        display(cb.metrics_table(evals))
        display(cb.size_table(evals))
else:
    print("Test no evaluado (EVALUAR_TEST = False).")

## 8. Comparación con el E1 en el mismo test

Es la comparación más limpia del cambio de dominio: **mismo test de CISUTAC, misma arquitectura y mismas métricas**; sólo cambian los datos de entrenamiento (CISUTAC en el E1, FabriD en el E4). Se leen las evaluaciones de test del E1 que guarda `notebooks/02_baseline_supervisado_cisutac.ipynb` en `results/cisutac/bloque_a/`; si todavía no existen, la tabla queda sólo con el E4.

La comparación con FabriD (sección 4) mide otra cosa: cuánto pierde el mismo modelo al salir de su dominio, mezclando el cambio de tela, de fotos y de escala.

In [ ]:
if EVALUAR_TEST:
    carpeta_e1 = ROOT / "results" / "cisutac" / "bloque_a" / "evaluaciones"
    e1 = [json.loads(p.read_text()) for p in sorted(carpeta_e1.glob("*__test_*/metricas.json"))]
    e1 = [m for m in e1 if "_prueba" not in m["corrida"]]
    if not e1:
        print(f"Todavía no hay evaluaciones de test del E1 en {carpeta_e1}.")
    for dataset, evals in test_cisutac.items():
        filas = [m for m in e1 if m["dataset"] == dataset] + evals
        print(dataset.upper())
        display(cb.metrics_table(filas))
        display(cb.size_table(filas))

## 9. Anexo exploratorio: FabricSpotDefect → CISUTAC Spots

> *Pendiente de aprobación del grupo:* no forma parte del E4 del protocolo. Sólo se ejecuta con `ANEXO_FSD = True`.

FabricSpotDefect son primeros planos de manchas sobre telas variadas: una segunda fuente industrial, sólo para Spots. `scripts/prepare_fsd_yolo.py` usa sólo los originales (no los aumentados) y corrige dos problemas que el EDA no había detectado:

- en 228 fotos, las cajas COCO corresponden a la foto girada 90° respecto del archivo; se pasan a la orientación del archivo;
- 8 fotos de train son copias, o la misma pieza con otro encuadre, de fotos de valid; como en FAB-4, pasan a valid.

También excluye las 36 fotos de train sin cajas, que no están verificadas como sanas. El entrenamiento y las evaluaciones son los mismos que con FabriD.

In [ ]:
if ANEXO_FSD:
    if not (tr.fsd_derived_dir() / "data.yaml").exists():
        subprocess.run([sys.executable, str(ROOT / "scripts" / "prepare_fsd_yolo.py")], check=True)
    resumen_fsd = json.loads((tr.fsd_derived_dir() / "resumen.json").read_text())
    print("Correcciones:", resumen_fsd["correcciones"])
    display(pd.DataFrame(resumen_fsd["splits"]).T)
    gt_fsd = tr.load_ground_truth(tr.fsd_derived_dir(), "train", "spot")
    lado_mayor = {im["id"]: max(im["width"], im["height"]) for im in gt_fsd["images"]}
    lados = sorted(a["area"] ** 0.5 * 640 / lado_mayor[a["image_id"]] for a in gt_fsd["annotations"])
    print(f"Lado mediano de las manchas de FabricSpotDefect al entrar al modelo: {lados[len(lados) // 2]:.1f} px")
else:
    print("Anexo no ejecutado (ANEXO_FSD = False).")

In [ ]:
if ANEXO_FSD:
    run_fsd = cb.train(f"e4fsd_spot_yolo11n_640{SUFIJO}", tr.fsd_derived_dir() / "data.yaml", imgsz=640,
                       epochs=EPOCAS, patience=PACIENCIA, batch=BATCH, seed=SEMILLA, model=MODELO,
                       retrain=REENTRENAR, fraction=FRACCION)
    PESOS_FSD = pesos(run_fsd)
    eval_fsd = evaluar_fsd(PESOS_FSD)
    evals_fsd_spots = [evaluar_cisutac("spots", PESOS_FSD, "completa", clave="spot", prefijo="e4fsd")] + [
        evaluar_cisutac("spots", PESOS_FSD, "tiles", tile=tile, clave="spot", prefijo="e4fsd") for tile in TILES]
    # Las corridas con FabriD (sección 5) se suman a la tabla si ya están.
    con_fabrid = evals_fsd_spots + (evals_cisutac["spots"] if "evals_cisutac" in globals() else [])
    display(cb.metrics_table([eval_fsd]))
    display(cb.metrics_table(con_fabrid))
    display(cb.size_table(con_fabrid))

    if EVALUAR_TEST:
        elegida_fsd = mejor_en_valid([m for m in evals_fsd_spots if m["modo"] == "tiles"])
        test_fsd = [evaluar_fsd(PESOS_FSD, split="test", umbral=eval_fsd["umbral_confianza"])] + [
            evaluar_cisutac("spots", PESOS_FSD, m["modo"], tile=m["tile"], split="test", umbral=m["umbral_confianza"],
                            clave="spot", prefijo="e4fsd") for m in (evals_fsd_spots[0], elegida_fsd)]
        display(cb.metrics_table(test_fsd))
        display(cb.size_table(test_fsd[1:]))

## 10. Conclusiones

> Completar **después** de correr el notebook sin `MODO_PRUEBA`, con los números de las tablas. No escribir conclusiones antes de tener resultados.

- **Escala o dominio:** ¿la foto completa no encuentra casi nada, como anticipa la sección 1? ¿Los tiles recuperan defectos? ¿Qué tamaño conviene para agujeros y para manchas, y por qué?
- **Caída respecto de FabriD:** mAP@0.5 de hole y stain en FabriD frente a Holes y Spots en CISUTAC, aclarando que cambian las fotos y la escala.
- **Brecha con el E1:** en el mismo test, ¿qué parte del rendimiento del E1 se alcanza sin ver ninguna prenda usada?
- **Errores típicos:** ¿qué confunde el modelo con defectos (costuras, estampados, ojalillos, pliegues)?
- **Anexo, si se aprueba:** ¿una fuente con manchas a una escala más parecida (FabricSpotDefect) transfiere mejor a Spots que FabriD?
- **Limitaciones:** una semilla; umbral y tile elegidos en valid de CISUTAC, así que no es una transferencia «a ciegas»; CISUTAC no tiene prendas sanas, de modo que los falsos positivos sólo se miden en prendas con defectos; el mapeo stain → mancha es aproximado; FabriD conserva su sesgo de lote.